# Multi-modal RAG with LangChain: Chat with PDFs

**Bonus notebook.** Prerequisite: tutorial 3 (LangChain), sections 7 and 8.

Tutorial 3 built RAG over plain text. Real PDFs also contain **tables** and **figures**, and a text loader destroys both.
Here we build a RAG chain that can retrieve all three and show a figure to a model that can see.

The problem: our embedding model understands only text, so it cannot embed a picture. The trick:

> **Embed a text summary of each element. Retrieve the original.**

```text
                    PDF
                     │  partition
        ┌────────────┼────────────┐
      texts        tables       images
        │            │            │
        └──── summarise each one ─┘
                     │
      summary → vector store      original → document store
                     └──── same doc_id ─────┘

question → search the summaries → fetch the originals → vision model → answer
```

This follows Alejandro AO's
[notebook](https://colab.research.google.com/gist/alejandro-ao/47db0b8b9d00b10a96ab42dd59d90b86/langchain-multimodal.ipynb)
and [video](https://www.youtube.com/watch?v=uLrReyH5cu0), with the paid parts replaced by free ones:

| Job | Original | Here |
|---|---|---|
| Read the PDF | `unstructured` (local) | `unstructured` (local) |
| Summarise text and tables | Llama 3.1 on Groq | `openai/gpt-oss-20b` on Groq |
| Describe images and answer | `gpt-4o-mini` (paid) | A vision model on Groq |
| Embeddings | `OpenAIEmbeddings` (paid) | `all-MiniLM-L6-v2` (local) |
| Vector store | Chroma | Chroma |

## Setup

Install the dependencies you need to run the notebook.

`unstructured` uses two **system programs**, which `pip` cannot install: **poppler** (turns PDF pages into images) and
**tesseract** (reads text in images). Google Colab is the easiest place to run this notebook.

In [ ]:
# for Linux and Google Colab
# !apt-get install -y poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

# for Windows (in a terminal with conda)
# conda install -c conda-forge poppler tesseract

In [ ]:
# %pip install -Uq "unstructured[pdf]" pillow lxml
# %pip install -Uq langchain langchain-classic langchain-groq
# %pip install -Uq langchain-chroma langchain-huggingface sentence-transformers
# %pip install -Uq python-dotenv

Uncomment the lines you need and run the two cells once. Then check that the system programs can be found:

In [ ]:
import shutil

for program in ["pdftoppm", "tesseract"]:
    print(f"{program:<10}", shutil.which(program) or "MISSING, see the installation cell above")

We need only one key, the free Groq key from tutorial 3. Put it in a `.env` file beside this notebook, as
`GROQ_API_KEY=your-key-here`. On Colab, create that file in the Files panel.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

TEXT_MODEL = "openai/gpt-oss-20b"    # reads text and tables
VISION_MODEL = "qwen/qwen3.6-27b"    # can also look at images

## Extract the data

Extract the elements of the PDF that we will use in the retrieval process: text, tables and images.

Our document is *Attention Is All You Need*, the paper that introduced the Transformer. It has an architecture diagram,
an attention diagram and several result tables. The file is in the `pdfs` folder of the course. On Colab, create a
`pdfs` folder in the Files panel and upload the file there.

In [ ]:
import pathlib

file_path = pathlib.Path("pdfs") / "attnetion_is_all_need.pdf"
assert file_path.exists(), f"{file_path} not found, put the PDF in the pdfs folder beside this notebook."

print(file_path)

### Partition PDF tables, text, and images

This is the slow cell: it takes several minutes on a CPU, because a layout model looks at every page. The first run
also downloads that model.

In [ ]:
from unstructured.partition.pdf import partition_pdf

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=str(file_path),
    infer_table_structure=True,            # extract tables
    strategy="hi_res",                     # mandatory to infer tables

    extract_image_block_types=["Image"],   # add "Table" to also get an image of each table
    extract_image_block_to_payload=True,   # keep images as base64, ready for the API

    chunking_strategy="by_title",          # or "basic"
    max_characters=4000,                   # a chunk is never longer than this
    combine_text_under_n_chars=1000,       # join very short sections
    new_after_n_chars=3000,                # prefer to start a new chunk after this
)

print(len(chunks), "chunks")

**`hi_res`** finds the boxes on each page and decides what each one is: a title, a paragraph, a table, an image.
**`by_title`** then groups the pieces into chunks that follow the paper's sections.

The original notebook uses chunks of up to 10,000 characters. Ours are smaller, because the free tier of Groq limits
the number of tokens per minute, and four long chunks plus images can exceed it.

We get two types of elements:

In [ ]:
set(type(chunk).__name__ for chunk in chunks)

Where are the images? Each `CompositeElement` is a **group** of related elements, and it keeps the members of the group in
`metadata.orig_elements`. The images are inside.

In [ ]:
def get_originals(chunk):
    return chunk.metadata.orig_elements or []

def is_image(element):
    return type(element).__name__ == "Image"

# the first chunk that contains an image
example = next(chunk for chunk in chunks if any(is_image(el) for el in get_originals(chunk)))

[type(el).__name__ for el in get_originals(example)]

And this is what one extracted image looks like. `image_base64` is there because we set
`extract_image_block_to_payload=True`. We shorten it for printing.

In [ ]:
example_image = next(el for el in get_originals(example) if is_image(el))

info = example_image.to_dict()
info["metadata"]["image_base64"] = info["metadata"]["image_base64"][:60] + "..."
info

### Separate extracted elements into tables, text, and images

In [ ]:
tables = []
texts = []

for chunk in chunks:
    kind = type(chunk).__name__
    if "Table" in kind:
        tables.append(chunk)
    elif "CompositeElement" in kind:
        texts.append(chunk)

In [ ]:
# Get the images from the CompositeElement objects
def get_images_base64(chunks):
    images_b64 = []
    for chunk in chunks:
        if "CompositeElement" in type(chunk).__name__:
            for el in get_originals(chunk):
                if is_image(el) and el.metadata.image_base64:
                    images_b64.append(el.metadata.image_base64)
    return images_b64

images = get_images_base64(chunks)

print(len(texts), "texts,", len(tables), "tables,", len(images), "images")

#### Check what the images look like

Always look. This crop is exactly what the vision model will see.

In [ ]:
import base64
from IPython.display import HTML, Image, display

def display_base64_image(base64_code):
    # Decode the base64 string to binary
    image_data = base64.b64decode(base64_code)
    # Display the image
    display(Image(data=image_data))

display_base64_image(images[0])

A table is kept as HTML, so every number stays in its own column:

In [ ]:
display(HTML(tables[0].metadata.text_as_html))

## Summarize the data

Create a summary of each element extracted from the PDF. The summary will be vectorized and used in the retrieval
process.

### Text and table summaries

We don't need a multimodal model for the tables and the text. A small open model on Groq is enough.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Prompt
prompt_text = """
You are an assistant tasked with summarizing tables and text.
Give a concise summary of the table or text.
Keep the important names, numbers and technical terms.

Respond only with the summary, no additional comment.
Do not start your message by saying "Here is a summary" or anything like that.
Just give the summary as it is.

Table or text chunk: {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
model = ChatGroq(model=TEXT_MODEL, temperature=0, max_retries=8)
summarize_chain = prompt | model | StrOutputParser()

`.batch` sends many inputs at once, and `max_concurrency` limits how many are in flight. With the free tier we go slowly.
If a request hits the rate limit, `max_retries` waits and tries again.

In [ ]:
# Summarize text
text_summaries = summarize_chain.batch([chunk.text for chunk in texts], {"max_concurrency": 2})

# Summarize tables
tables_html = [table.metadata.text_as_html for table in tables]
table_summaries = summarize_chain.batch(tables_html, {"max_concurrency": 2})

In [ ]:
print("TEXT: ", text_summaries[0])
print()
print("TABLE:", table_summaries[0])

### Image summaries

The original uses `gpt-4o-mini`. We use a vision model on Groq instead.

Until now a message was a string. A **multimodal message** is a list of blocks: a text block and an image block. The image
travels inside the request as a *data URI*, `data:image/jpeg;base64,...`.

In [ ]:
vision_model = ChatGroq(
    model=VISION_MODEL,
    temperature=0,
    max_retries=8,
    reasoning_effort="none",   # Qwen setting: answer directly. Remove it for other models.
)

prompt_template = """Describe the image in detail. For context,
the image is part of a research paper explaining the transformers
architecture. Be specific about graphs, such as bar plots."""

messages = [
    (
        "user",
        [
            {"type": "text", "text": prompt_template},
            {
                "type": "image_url",
                "image_url": {"url": "data:image/jpeg;base64,{image}"},
            },
        ],
    )
]

image_prompt = ChatPromptTemplate.from_messages(messages)
image_chain = image_prompt | vision_model | StrOutputParser()

image_summaries = image_chain.batch([{"image": b64} for b64 in images], {"max_concurrency": 1})

In [ ]:
display_base64_image(images[0])
print(image_summaries[0])

Compare the picture with its description. Search will see **only the description**, so a detail that is missing there
cannot be found later.

## Load data and summaries to vectorstore

### Create the vectorstore

We need two stores, and **`MultiVectorRetriever`** connects them:

- the **vector store** holds the summaries, and is searched;
- the **document store** holds the originals, and is what you get back;
- **`doc_id`** links a summary to its original, like a cloakroom ticket.

In [ ]:
import uuid
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# The vectorstore to use to index the summaries
vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=embeddings)
vectorstore.reset_collection()   # start empty, so running this cell twice is safe

# The storage layer for the originals
store = InMemoryStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
    search_kwargs={"k": 4},
)

### Load the summaries and link them to the original data

The same three steps for texts, tables and images: make one id per element, store the summaries with their ids, and store
the originals under the same ids.

In [ ]:
# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=summary, metadata={id_key: doc_ids[i]}) for i, summary in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

In [ ]:
# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=summary, metadata={id_key: table_ids[i]}) for i, summary in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

In [ ]:
# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in images]
summary_img = [
    Document(page_content=summary, metadata={id_key: img_ids[i]}) for i, summary in enumerate(image_summaries)
]
retriever.vectorstore.add_documents(summary_img)
retriever.docstore.mset(list(zip(img_ids, images)))

### Check retrieval

Test the retriever before adding a model. The results are originals: a base64 string for an image, an element for
anything else.

In [ ]:
docs = retriever.invoke("who are the authors of the paper?")

for doc in docs:
    print("IMAGE" if isinstance(doc, str) else str(doc)[:500])
    print("-" * 80)

## RAG pipeline

The same shape as tutorial 3, with one new problem: the retrieved items are mixed. Text goes into the prompt, and images
must be attached as image blocks. We build the chain in three small parts.

### Part 1: sort the retrieved items

In [ ]:
def parse_docs(docs):
    """Split base64-encoded images and texts"""
    b64 = []
    text = []
    for doc in docs:
        if isinstance(doc, str):     # we stored images as base64 strings
            b64.append(doc)
        else:                        # texts and tables are unstructured elements
            text.append(doc)
    return {"images": b64, "texts": text}

found = parse_docs(retriever.invoke("Show me the architecture of the Transformer"))
print(len(found["texts"]), "texts or tables,", len(found["images"]), "images")

### Part 2: build the prompt

Tables go in as HTML, so the model can see which number belongs to which column. We attach at most two images, because
images use many tokens and the API limits the number per request.

In [ ]:
from langchain_core.messages import HumanMessage

MAX_IMAGES = 2

def element_to_text(element):
    if "Table" in type(element).__name__:
        return element.metadata.text_as_html
    return element.text

def build_prompt(kwargs):
    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = "\n\n".join(element_to_text(el) for el in docs_by_type["texts"])

    # construct prompt with context (including images)
    prompt_text = f"""
    Answer the question based only on the following context, which can include text, tables, and the images below.
    If the answer is not in the context, say that you do not know.
    Context: {context_text}
    Question: {user_question}
    """

    prompt_content = [{"type": "text", "text": prompt_text}]

    for image in docs_by_type["images"][:MAX_IMAGES]:
        prompt_content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            }
        )

    return [HumanMessage(content=prompt_content)]

### Part 3: connect everything

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

chain = (
    {
        "context": retriever | RunnableLambda(parse_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_prompt)
    | vision_model
    | StrOutputParser()
)

In [ ]:
response = chain.invoke("What is the attention mechanism?")

print(response)

### The same chain, with sources

Never trust a RAG answer you cannot check. `RunnablePassthrough.assign` keeps the context and **adds** the response, so we
get both.

In [ ]:
chain_with_sources = {
    "context": retriever | RunnableLambda(parse_docs),
    "question": RunnablePassthrough(),
} | RunnablePassthrough.assign(
    response=(
        RunnableLambda(build_prompt)
        | vision_model
        | StrOutputParser()
    )
)

In [ ]:
def show_answer(result):
    print("Response:", result["response"])
    print("\n\nContext:")
    for element in result["context"]["texts"]:
        print(element.text[:500])
        print("Page number: ", element.metadata.page_number)
        print("\n" + "-" * 50 + "\n")
    for image in result["context"]["images"][:MAX_IMAGES]:
        display_base64_image(image)

show_answer(chain_with_sources.invoke("What is multihead?"))

Now a question about a figure, and one about a table:

In [ ]:
show_answer(chain_with_sources.invoke("Describe the diagram of the Transformer model architecture."))

In [ ]:
show_answer(chain_with_sources.invoke("What BLEU score did the Transformer (big) get on English to German?"))

**Be careful when judging the answers.** Every large model has seen this famous paper in training, so a correct answer
does not prove that retrieval worked. Check the printed context. For a fair test, use a PDF that the model cannot know,
such as your own report.

## Exercises

1. Ask a question about something that is not in the paper. Does the model admit it?
2. Change the image prompt to "Describe the image briefly." and index again. Is the architecture figure still found?
3. Set `search_kwargs={"k": 2}`, then `{"k": 6}`. What happens to the answers and to the rate limit?
4. Replace the PDF with your own.

## Glossary

| Term | Meaning |
|---|---|
| **Multi-modal RAG** | RAG over text, tables and images. |
| **`partition_pdf`** | Splits a PDF into typed elements. `hi_res` uses a layout model. |
| **`CompositeElement`** | A chunk made of several elements, kept in `orig_elements`. |
| **`text_as_html`** | A table with its rows and columns preserved. |
| **Base64** | A way to write an image as text. |
| **Multimodal message** | A message whose content is a list of text and image blocks. |
| **`MultiVectorRetriever`** | Searches summaries and returns the originals. |
| **`doc_id`** | The ticket that links a summary to its original. |

## References

- [The original notebook](https://colab.research.google.com/gist/alejandro-ao/47db0b8b9d00b10a96ab42dd59d90b86/langchain-multimodal.ipynb) by Alejandro AO
- [LangChain inspiration](https://github.com/langchain-ai/langchain/blob/master/cookbook/Semi_structured_and_multi_modal_RAG.ipynb)
- [Multi-vector storage](https://python.langchain.com/docs/how_to/multi_vector/)